# Qualitative behaviour of the two frozen encoders

For the same cross-matched objects: per-head `[CLS]` attention of the image
encoder's final block, and the masked reconstruction of the spectrum encoder.
Writes `evaluation.png`.

In [ ]:
import sys, gc
import numpy as np
import torch
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.table import Table

from plotstyle import (ROOT, IMAGE_DIR, SPEC_DIR, SPEC_OUT, IMAGE_ROOT, DJA_FITS, XMATCH,
                       IMG_CKPT, C_ORIG, C_IMAGE, C_RED, C_RED_BRIGHT, use_style, style_axes, fs, scalebar, save)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SPEC_CKPT = sorted((SPEC_OUT / "low_res_pt_1_2_micron_noz_cut"
                    / "version_0/checkpoints").glob("epoch=*.ckpt"))[-1]

N_POOL = 40    # candidates kept after the extended-source cut; ROWS indexes into these
# Tokens hidden per prediction. The tokenizer overlaps (patch 4, stride 2), so a
# neighbour still carries the target's pixels: block_k=3 is the leak-free version.
BLOCK_K = 3

# Rest-frame emission lines (micron) annotated on the reconstruction panels.
EMISSION_LINES = {
    r"Ly$\alpha$": 0.1216, "[OII]": 0.3727, r"H$\beta$": 0.4861, "[OIII]": 0.5007,
    r"H$\alpha$": 0.6563, "[SII]": 0.6724, r"Pa$\zeta$": 0.9229,
    r"Pa$\epsilon$+[SIII]": 0.9539, r"Pa$\delta$": 1.0049, r"Pa$\gamma$": 1.0938,
}

use_style(1.45)
print(f"device {DEVICE}\nimage    {IMG_CKPT.name}\nspectrum {SPEC_CKPT.name}")

## 1. Cross-matched sample

Real galaxies with a well-measured, gap-free spectrum inside the encoder's window.

In [ ]:
xt = Table.read(XMATCH)
with fits.open(DJA_FITS, memmap=False) as hdul:
    wave_full = np.asarray(hdul["WAVE"].data, dtype=np.float32)
    cat = hdul["CATALOG"].data
    flux_all = np.asarray(cat["flux"], dtype=np.float32)        # f_nu, uJy
    valid_all = np.asarray(cat["valid_spec"], dtype=bool)
    sn50_all = np.asarray(cat["sn50"], dtype=np.float32)
    z_all = np.asarray(cat["z_best"], dtype=np.float32)

dja_id = np.asarray(xt["dja_id"], dtype=np.int64)
wl_keep = (wave_full > 1.0) & (wave_full < 2.0)                 # training window
wave_win = wave_full[wl_keep]

flux_win = flux_all[dja_id][:, wl_keep]
finite = np.isfinite(flux_win)
flux_win = np.where(finite, flux_win, 0.0).astype(np.float32)
valid_win = valid_all[dja_id][:, wl_keep] & finite
flux_win = flux_win / wave_win[None, :] ** 2                    # f_nu -> f_lambda

n_valid = valid_win.sum(axis=1)
sn50, zz = sn50_all[dja_id], z_all[dja_id]

# one row per spectrum, then: a real galaxy, well measured, no gaps
_, first = np.unique(dja_id, return_index=True)
ok = first[(zz[first] > 0.4) & (zz[first] < 3.5) & np.isfinite(sn50[first])
           & (sn50[first] > 40) & (n_valid[first] == len(wave_win))]
ranked = ok[np.argsort(-sn50[ok])]
print(f"{len(xt)} crossmatch rows -> {len(ranked)} candidates")

## 2. Image encoder — per-head `[CLS]` attention

In [ ]:
sys.path.insert(0, str(IMAGE_DIR))
from model.jwst_dino import JWST_DINO                      # noqa: E402
from model.modules import PatchEmbed                       # noqa: E402
from data.augmentations import AsinhStretch                # noqa: E402
from torchvision.transforms.functional import center_crop  # noqa: E402

img_model = JWST_DINO.load_from_checkpoint(str(IMG_CKPT), map_location=DEVICE).eval().to(DEVICE)
net = img_model.teacher_backbone
CROP = img_model.hparams.global_crops_size
SIDE = PatchEmbed.num_tokens_per_side(CROP, net.patch_size, net.patch_stride)
HEADS, NREG = net.blocks[0].attn.num_heads, net.num_register_tokens

stretch = AsinhStretch(return_channel_pos=0)
_shards = {}


def load_crop(i):
    """Row i of the crossmatch -> (1,1,CROP,CROP) exactly as the encoder sees it."""
    rel, loc = str(xt["rel_path"][i]), int(xt["local_idx"][i])
    if rel not in _shards:
        _shards[rel] = np.load(IMAGE_ROOT / rel, mmap_mode="r")
    raw = np.nan_to_num(np.asarray(_shards[rel][loc], dtype=np.float32))
    c = center_crop(torch.from_numpy(raw)[None, None], CROP)
    return torch.from_numpy(np.asarray(stretch(c[0].numpy()))).unsqueeze(0).to(DEVICE)


def block_attention(x, block=-1):
    """(1,1,H,W) -> (HEADS, SIDE, SIDE) CLS->patch attention of one block.

    Attention.forward uses F.scaled_dot_product_attention, which never materialises
    the weights, so a hook on `qkv` recomputes just the CLS row of softmax(qk^T).
    Register columns stay inside the softmax (they compete for mass, as at training
    time) and are dropped afterwards, so what is plotted is the patch share.
    """
    blk = net.blocks[block]
    grabbed = {}
    h = blk.attn.qkv.register_forward_hook(
        lambda m, i, o: grabbed.__setitem__("qkv", o.detach()))
    try:
        with torch.no_grad():
            net(x)
    finally:
        h.remove()
    qkv = grabbed["qkv"]
    B, T, _ = qkv.shape
    hd = blk.attn.head_dim
    q, k, _ = qkv.reshape(B, T, 3, HEADS, hd).permute(2, 0, 3, 1, 4)
    a = ((q[:, :, :1] @ k.transpose(-2, -1)) * hd ** -0.5).softmax(-1)[0, :, 0]
    return a[:, 1 + NREG:].reshape(HEADS, SIDE, SIDE).cpu().numpy()


def is_extended(img):
    """Reject point sources: a star puts most of its light in the central 12x12 px."""
    tot = img.sum()
    return tot > 0 and img[CROP // 2 - 6:CROP // 2 + 6,
                           CROP // 2 - 6:CROP // 2 + 6].sum() / tot < 0.35


# S/N-ranked pool of extended sources, plus three hand-picked non-JADES objects
# (high ellipticity, clear emission lines) appended at the end.
gals = []
for i in ranked:
    if is_extended(load_crop(int(i))[0, 0].cpu().numpy()):
        gals.append(int(i))
    if len(gals) == N_POOL:
        break
EXTRA = [5173, 3056, 7522]
sel = np.array([g for g in gals if g not in EXTRA] + EXTRA)

# DINO's choice: the final block, every head kept separate.
attn = np.stack([block_attention(load_crop(int(i)), block=-1) for i in sel])
crops = np.stack([load_crop(int(i))[0, 0].cpu().numpy() for i in sel])
# The token grid is stretched over the crop via `extent`; np.kron would misalign
# it because CROP is not an exact multiple of SIDE.
EXTENT = [-0.5, CROP - 0.5, -0.5, CROP - 0.5]
print(f"{crops.shape[0]} objects  {SIDE}x{SIDE} tokens  {HEADS} heads")

## 3. Spectrum encoder — masked reconstruction

Every token is hidden in turn and predicted from the rest, so nothing sees itself.

In [ ]:
del img_model, net
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
for m in [m for m in sys.modules if m in ("model", "data") or m.startswith(("model.", "data."))]:
    del sys.modules[m]
sys.path.insert(0, str(SPEC_DIR))

from model.low_res_pt import LowResPT                      # noqa: E402

spec = LowResPT.load_from_checkpoint(str(SPEC_CKPT), map_location=DEVICE).eval().to(DEVICE)
if "min_std" not in spec.hparams:                          # older ckpts predate it
    spec.hparams["min_std"] = 0.1
P, S = spec.hparams.patch_size, spec.hparams.stride


def unpatchify(patches, L_used):
    """(N,P) overlapping patches -> (L_used,) by averaging the overlaps."""
    out, cnt = np.zeros(L_used), np.zeros(L_used)
    for n in range(patches.shape[0]):
        sl = slice(n * S, n * S + P)
        out[sl] += patches[n]
        cnt[sl] += 1
    return np.where(cnt > 0, out / np.maximum(cnt, 1), np.nan)


recons = []
for i in sel:
    f = torch.from_numpy(flux_win[i])[None].to(DEVICE)
    w = torch.from_numpy(wave_win)[None].to(DEVICE)
    v = torch.from_numpy(valid_win[i])[None].to(DEVICE)
    r = spec.reconstruct(f, w, v, masked=True, block_k=BLOCK_K)
    L = int(r["L_used"])
    recons.append({"wave": wave_win[:L],
                   "input": r["flux_norm"][0, :L].cpu().numpy(),
                   "recon": unpatchify(r["recon_patches"][0].cpu().numpy(), L),
                   "valid": valid_win[i][:L]})
print(f"{len(recons)} spectra, {L} px each")

## 4. Figure

In [ ]:
# One row per object at fixed height H: cutout | 2x4 head grid | spectrum. Column
# widths 1 : 2 : 1.5 keep every attention panel square and the grid inside the row.
ROWS = [22, 9, 20, 8, 21]     # indexes into `sel`
H = 3.0
LABEL_C = C_RED_BRIGHT
RECON_C = C_IMAGE

fig = plt.figure(figsize=(4.5 * H, H * len(ROWS)))
outer = fig.add_gridspec(len(ROWS), 3, width_ratios=[1, 2, 1.5], wspace=0.12, hspace=0.10)
ax_ref = None

for r_, r in enumerate(ROWS):
    i, rec = sel[r], recons[r]
    lo, hi = np.percentile(crops[r], [1, 99.5])

    # --- cutout ---
    ax0 = fig.add_subplot(outer[r_, 0])
    ax0.imshow(crops[r], cmap="gray", vmin=lo, vmax=hi, origin="lower")
    ax0.set_xticks([])
    ax0.set_yticks([])
    ax0.text(0.035, 0.965, f"{str(xt['survey'][i])}\n{dja_id[i]}", transform=ax0.transAxes,
             fontsize=fs(11), color=LABEL_C, fontweight="bold", va="top", ha="left",
             linespacing=1.3)
    scalebar(ax0, n_pix=20, fontsize=fs(10))

    # --- per-head attention, final block ---
    inner = outer[r_, 1].subgridspec(2, 4, wspace=0.04, hspace=0.04)
    for h in range(HEADS):
        ax = fig.add_subplot(inner[h // 4, h % 4])
        ax.imshow(crops[r], cmap="gray", vmin=lo, vmax=hi, origin="lower")
        ax.imshow(attn[r, h], cmap="hot", alpha=0.65, origin="lower", extent=EXTENT,
                  interpolation="nearest")
        ax.axis("off")
        ax.text(0.04, 0.94, f"{h}", transform=ax.transAxes, fontsize=7, color="w",
                va="top", ha="left")

    # --- spectrum: input vs masked reconstruction ---
    ax = fig.add_subplot(outer[r_, 2], sharex=ax_ref)
    ax_ref = ax_ref or ax
    m = rec["valid"]
    ax.plot(rec["wave"], np.where(m, rec["input"], np.nan), color=C_RED, lw=1.5,
            label="input")
    ax.plot(rec["wave"], np.where(m, rec["recon"], np.nan), color=RECON_C, lw=1.5,
            ls="--", label="reconstruction")
    for k in np.flatnonzero(~m):
        ax.axvspan(rec["wave"][k] - 0.009, rec["wave"][k] + 0.009, color="0.6",
                   alpha=0.25, lw=0)

    # emission lines in the observed frame; headroom is added first so the
    # alternating labels sit inside the panel, and edge lines are aligned inwards
    wlo, whi = rec["wave"][0], rec["wave"][-1]
    ylo, yhi = ax.get_ylim()
    yhi += 0.30 * (yhi - ylo)
    ax.set_ylim(ylo, yhi)
    lev = [yhi - 0.03 * (yhi - ylo), yhi - 0.15 * (yhi - ylo)]
    vis = sorted([(nm, wl * (1 + zz[i])) for nm, wl in EMISSION_LINES.items()
                  if wlo <= wl * (1 + zz[i]) <= whi], key=lambda t: t[1])
    for ei, (nm, wl) in enumerate(vis):
        ax.axvline(wl, color="#999999", ls="--", lw=1.0, alpha=0.8)
        f = (wl - wlo) / (whi - wlo)
        ax.text(wl, lev[ei % 2], nm, va="top", fontsize=fs(8), color="#333333",
                ha="left" if f < 0.08 else ("right" if f > 0.92 else "center"),
                fontweight="bold",
                bbox=dict(facecolor="white", alpha=0.75, pad=1.5, edgecolor="none"))
    ax.set_xlim(wlo, whi)
    if r_ == len(ROWS) // 2:
        ax.set_ylabel("norm. flux", fontsize=fs(8), labelpad=4)
    style_axes(ax)
    if r_ == 0:
        ax.legend(fontsize=fs(6.5), loc="lower left", framealpha=0.85, edgecolor="none",
                  handlelength=1.4, handletextpad=0.4, borderpad=0.3, labelspacing=0.3)
    if r_ == len(ROWS) - 1:
        ax.set_xlabel(r"observed $\lambda$  ($\mu$m)", fontsize=fs(12))
    else:
        ax.tick_params(labelbottom=False)

fig.patch.set_alpha(0.0)
save(fig, "evaluation")
plt.show()